# Feature Engineering :

De données prétraitées à des variables adaptées aux deux approches d'apprentissage. **Point de départ : les données nettoyées et sauvegardées après le preprocessing.**

Deux volets sont considérés : l'apprentissage supervisé (prédiction de la température du mois suivant à partir des variables météorologiques, du mois et de l'année) et l'apprentissage non supervisé (extraction de caractéristiques à partir de la température et du rayonnement thermique pour étudier les tendances climatiques à long terme).

Ordre des étapes : création des cibles (température du mois suivant et classes Low/High) → encodage cyclique (Month Sin/Cos) → transformation SignedLog1p → RobustScaler → préparation des différents ensembles de variables → sauvegarde des données transformées.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import RobustScaler
import os, numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis
BASE    = r"D:\M2\Data Mining\DM-Weather-framework\data"
OUT_DIR = f"{BASE}/processed/transformed"
HEAVY = ["slhf", "e", "ro", "tp"]

In [42]:

# Chargement des datasets
train = pd.read_parquet(f"{OUT_DIR}/train_transformed.parquet")
test = pd.read_parquet(f"{OUT_DIR}/test_transformed.parquet")
df = pd.read_parquet(f"{OUT_DIR}/dataset_unsupervised.parquet")

print("Train :", train.shape)
print("Test :", test.shape)

Train : (596750, 18)
Test : (151900, 18)


In [ ]:
train.head(1)

,time,latitude,longitude,t2m,d2m,sp,u10,v10,slhf,sshf,ssr,str,e,ro,tp,month,t2m_next,target_time
0,2015-01-01,18.95,3.35,16.053369,-7.53208,96317.875,-3.543621,-2.430153,-0.267799,-1886500.0,11383417.0,-9657388.0,-0.267801,0.0,0.062363,1,21.763574,2015-02-01


## **Apprentissage supervise:**

### 1. Création des variables temporelle et cyclique

In [43]:
# Puisque ta colonne month existe déjà, il suffit de créer la colonne year à partir de time.
for data in [train, test]:
    data["year"] = data["time"].dt.year

In [44]:
for data in [train, test]:

    data["month_sin"] = np.sin(
        2 * np.pi * data["month"] / 12)

    data["month_cos"] = np.cos(
        2 * np.pi * data["month"] / 12)

In [45]:
FEATURES_CYCLIC = [
    "d2m", "slhf", "ssr", "str", "sshf",
    "ro", "e", "u10", "v10", "sp", "tp","year",
    "month_sin", "month_cos"
]

TARGET = "t2m_next"

X_train = train[FEATURES_CYCLIC].copy()
y_train = train[TARGET].copy()

X_test = test[FEATURES_CYCLIC].copy()
y_test = test[TARGET].copy()

print("X_train :", X_train.shape)
print("y_train :", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

X_train : (596750, 14)
y_train : (596750,)
X_test : (151900, 14)
y_test : (151900,)


### 2. Normalisation avec RobustScaler

In [46]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=FEATURES_CYCLIC,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=FEATURES_CYCLIC,
    index=X_test.index
)

### 3. Sauvgarde:

In [47]:
import os
import joblib

SUP_DIR = os.path.join(OUT_DIR, "supervised")

os.makedirs(SUP_DIR, exist_ok=True)

# Préparation des datasets finaux
train_supervised = X_train_scaled.copy()
train_supervised[TARGET] = y_train

test_supervised = X_test_scaled.copy()
test_supervised[TARGET] = y_test

# Sauvegarde
train_supervised.to_parquet(
    os.path.join(SUP_DIR, "train_supervised.parquet"),
    index=False
)

test_supervised.to_parquet(
    os.path.join(SUP_DIR, "test_supervised.parquet"),
    index=False
)

# Sauvegarde du scaler
joblib.dump(
    scaler,
    os.path.join(SUP_DIR, "robust_scaler.pkl")
)

print("========== SAUVEGARDE SUPERVISÉE ==========")
print("Train :", train_supervised.shape)
print("Test :", test_supervised.shape)
print("Dossier :", SUP_DIR)

========== SAUVEGARDE SUPERVISÉE ==========
Train : (596750, 15)
Test : (151900, 15)
Dossier : /content/drive/MyDrive/DataMining_Climat/processed/transformed/supervised


## **Apprentissage supervise:**

In [48]:
# Regrouper les données Train et Test
df_unsup = df.copy()

# Sélectionner uniquement les variables nécessaires
df_unsup = df_unsup[
    ["time", "latitude", "longitude", "t2m", "str"]
].copy()

# Conversion de la date
df_unsup["time"] = pd.to_datetime(df_unsup["time"])

# Extraction de l'année et du mois
df_unsup["year"] = df_unsup["time"].dt.year
df_unsup["month"] = df_unsup["time"].dt.month

# Vérification
print("Dimensions :", df_unsup.shape)
print("Période :", df_unsup["time"].min(), "à", df_unsup["time"].max())

df_unsup.head()

Dimensions : (754075, 7)
Période : 2015-01-01 00:00:00 à 2026-07-01 00:00:00


,time,latitude,longitude,t2m,str,year,month
0,2015-01-01,18.95,3.35,16.053369,-9657388.0,2015,1
1,2015-02-01,18.95,3.35,21.763574,-10289442.0,2015,2
2,2015-03-01,18.95,3.35,24.019189,-10270548.0,2015,3
3,2015-04-01,18.95,3.35,27.534326,-12807040.0,2015,4
4,2015-05-01,18.95,3.35,33.414697,-12867167.0,2015,5


### 1. Vérifier les observations
Avant le pivot, vérifions que nous avons une seule observation par combinaison de date et de coordonnées.

**Absence de doublons :**
- Nombre de doublons : 0.
- Chaque combinaison time, latitude, longitude est unique.
- Nous pouvons donc utiliser pivot sans risque de rencontrer des combinaisons dupliquées.

**Régularité des observations :**
- Nombre de mois : 138.
- Nombre d'observations par mois : 5 425.
- Écart-type : 0.

Cela signifie que chaque mois possède exactement les mêmes 5 425 points géographiques. La couverture spatiale est donc constante pendant toute la période étudiée. Or, La combinaison (latitude, longitude, time) est unique, donc pivot() convient.

In [49]:
# Vérifier les doublons
n_duplicates = df_unsup.duplicated(
    subset=["time", "latitude", "longitude"]
).sum()

print("Nombre de doublons :", n_duplicates)

# Nombre d'observations par mois
monthly_counts = df_unsup.groupby("time").size()

print("\nNombre d'observations par mois :")
print(monthly_counts.describe())

Nombre de doublons : 0

Nombre d'observations par mois :
count     139.0
mean     5425.0
std         0.0
min      5425.0
25%      5425.0
50%      5425.0
75%      5425.0
max      5425.0
dtype: float64


In [50]:
# Vérifier les observations par année et par mois
monthly_year_counts = (
    df_unsup.groupby(["year", "month"])
    .size()
    .unstack(fill_value=0)
)

print(monthly_year_counts)

month    1     2     3     4     5     6     7     8     9     10    11    12
year                                                                         
2015   5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425
2016   5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425
2017   5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425
2018   5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425
2019   5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425
2020   5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425
2021   5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425
2022   5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425
2023   5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425
2024   5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425  5425
2025   5425  5425  5425  5425  5425  5425  5425  5425  5425  542

### Transformer les données avec Pivot

In [51]:
# Transformation au format large
df_wide = df_unsup.pivot(
    index=["latitude", "longitude", "year"],
    columns="month",
    values=["t2m", "str"]
)

# Renommer les colonnes
df_wide.columns = [
    f"{var}_month_{month:02d}"
    for var, month in df_wide.columns
]

# Réinitialiser les index
df_wide = df_wide.reset_index()

df_wide.head()

,latitude,longitude,year,t2m_month_01,t2m_month_02,t2m_month_03,t2m_month_04,t2m_month_05,t2m_month_06,t2m_month_07,...,str_month_03,str_month_04,str_month_05,str_month_06,str_month_07,str_month_08,str_month_09,str_month_10,str_month_11,str_month_12
0,18.95,3.35,2015,16.053369,21.763574,24.019189,27.534326,33.414697,35.476709,34.426172,...,-10270548.0,-12807040.0,-12867167.0,-12511503.0,-9670138.0,-9527995.0,-10247684.0,-10109498.0,-9816150.0,-9350701.0
1,18.95,3.35,2016,18.776025,21.222314,24.507715,32.480615,35.356104,35.463281,34.271631,...,-11253493.0,-11530312.0,-11487663.0,-11117743.0,-10337418.0,-9838213.0,-10950351.0,-10846073.0,-9787626.0,-9044229.0
2,18.95,3.35,2017,16.230127,21.575342,26.510889,30.145898,34.917871,34.681543,34.560938,...,-11706741.0,-12207385.0,-11359854.0,-10168740.0,-10375005.0,-9834490.0,-10946946.0,-11031628.0,-10162840.0,-9069576.0
3,18.95,3.35,2018,17.994043,21.937158,26.255029,30.052637,34.361230,35.861475,34.570215,...,-12037101.0,-12616256.0,-12097719.0,-11510019.0,-10949318.0,-8719697.0,-9987941.0,-10226850.0,-9647066.0,-9640336.0
4,18.95,3.35,2019,16.611475,18.534326,25.455225,29.940332,34.126855,35.696191,35.222803,...,-10880568.0,-11696590.0,-12408240.0,-11175396.0,-10335122.0,-8164468.0,-10771665.0,-9936483.0,-9809740.0,-9916268.0


In [52]:
# Nombre de mois disponibles par année
df_wide["nb_mois"] = df_wide.drop(
    columns=["latitude", "longitude", "year"]
).notna().sum(axis=1)

print(df_wide.groupby("year")["nb_mois"].describe())

       count  mean  std   min   25%   50%   75%   max
year                                                 
2015  5425.0  24.0  0.0  24.0  24.0  24.0  24.0  24.0
2016  5425.0  24.0  0.0  24.0  24.0  24.0  24.0  24.0
2017  5425.0  24.0  0.0  24.0  24.0  24.0  24.0  24.0
2018  5425.0  24.0  0.0  24.0  24.0  24.0  24.0  24.0
2019  5425.0  24.0  0.0  24.0  24.0  24.0  24.0  24.0
2020  5425.0  24.0  0.0  24.0  24.0  24.0  24.0  24.0
2021  5425.0  24.0  0.0  24.0  24.0  24.0  24.0  24.0
2022  5425.0  24.0  0.0  24.0  24.0  24.0  24.0  24.0
2023  5425.0  24.0  0.0  24.0  24.0  24.0  24.0  24.0
2024  5425.0  24.0  0.0  24.0  24.0  24.0  24.0  24.0
2025  5425.0  24.0  0.0  24.0  24.0  24.0  24.0  24.0
2026  5425.0  14.0  0.0  14.0  14.0  14.0  14.0  14.0


L'objectif est de construire des profils annuels complets, donc c'est mieux de conserver uniquement les années 2015 à 2025 :

In [53]:
# Conserver les années complètes
df_wide_complete = df_wide[df_wide["year"] < 2026].copy()

print("Dimensions :", df_wide_complete.shape)
print("Années :", sorted(df_wide_complete["year"].unique()))
print("Valeurs manquantes :", df_wide_complete.isnull().sum().sum())

Dimensions : (59675, 28)
Années : [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
Valeurs manquantes : 0


### 2. Creation des caractéristiques statistiques annuelles:

In [54]:
# Variables mensuelles
t2m_cols = [f"t2m_month_{m:02d}" for m in range(1, 13)]
str_cols = [f"str_month_{m:02d}" for m in range(1, 13)]

# Statistiques annuelles de température
df_wide_complete["t2m_mean"] = df_wide_complete[t2m_cols].mean(axis=1)
df_wide_complete["t2m_std"] = df_wide_complete[t2m_cols].std(axis=1)
df_wide_complete["t2m_min"] = df_wide_complete[t2m_cols].min(axis=1)
df_wide_complete["t2m_max"] = df_wide_complete[t2m_cols].max(axis=1)
df_wide_complete["t2m_amplitude"] = (
    df_wide_complete["t2m_max"] - df_wide_complete["t2m_min"]
)

# Statistiques annuelles de rayonnement
df_wide_complete["str_mean"] = df_wide_complete[str_cols].mean(axis=1)
df_wide_complete["str_std"] = df_wide_complete[str_cols].std(axis=1)
df_wide_complete["str_min"] = df_wide_complete[str_cols].min(axis=1)
df_wide_complete["str_max"] = df_wide_complete[str_cols].max(axis=1)
df_wide_complete["str_amplitude"] = (
    df_wide_complete["str_max"] - df_wide_complete["str_min"]
)